In [2]:
import warnings
import numpy as np
import os
import glob
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning, module="openpyxl")

folder_path = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Dependency_Jeeves\Consolidated_Python"
excel_files = glob.glob(os.path.join(folder_path, "*.xlsx"))

dataframes_with_dates = []

# Iterate over all Excel files
for file in excel_files:
    # Read the Excel file into a DataFrame
    df = pd.read_excel(file, engine="openpyxl")
    
    date_str = os.path.basename(file).split('_')[0].replace('.xlsx', '')
    
    date_formatted = pd.to_datetime(date_str).strftime('%Y-%m-%d')
    
    df['Date'] = date_formatted
    dataframes_with_dates.append(df)


combined_df = pd.concat(dataframes_with_dates, ignore_index=True) if dataframes_with_dates else pd.DataFrame()
print(combined_df.tail(10))



                             CASE_ID           Repair Call Status  \
0  FLRM-BLH8M3-JGEHHL-IAZ7MB-RNJRB03            SERVICE_ALLOCATED   
1  FLRM-C305V6-IJJRNZ-4RMCUP-H36Z3M6            SERVICE_ALLOCATED   
2  SPXT-8MCNNJ-ZAT30T-OMUASX-BQOPDQ5              PART_IN_TRANSIT   
3  FLRM-3CDUX0-W12GZB-PFYJOI-19H9L0M            SERVICE_ALLOCATED   
4  FLRR-9RRNG5-85NYD2-RBVIYG-L699N1X         ALLOCATED_WITH_PARTS   
5  FLRM-BJZVGT-70ZT0J-P51TYP-ZI7EEFM            SERVICE_ALLOCATED   
6  FLRM-3DOAND-H5P2KW-Q1R47F-PE9GOZS            SERVICE_ALLOCATED   
7  FLRM-EP8KSZ-2S5RQV-P173KW-RSQGY9K  KPO_CUSTOMER_NOT_RESPONDING   
8  FLRM-5SAP6Y-AK5CU2-PK0RRR-6ELU4Q9            SERVICE_ALLOCATED   
9   FLRR-QDA3C4-AZDU9U-QTR135-3PRDP6       KPO_CUSTOMER_NOT_READY   

         Part Status DEPENDENCY  CALL_AGEING CALL_AGEING_BAND  STATUS_AGEING  \
0                NaN        OPS           12          g.11~15              2   
1                NaN        OPS            2            c.2~3              1   


In [7]:
Attribute_map = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Dependency_Jeeves\Attribution_Mapping.xlsx"
df_Attribute = pd.read_excel(Attribute_map)

Aging_map = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Dependency_Jeeves\Breach_aging.xlsx"
df_aging = pd.read_excel(Aging_map)

plan_name = r"G:\My Drive\New_Working_Files\KNIME\CTU_2025\Dependency_Jeeves\Plan_Names.xlsx"
df_plan = pd.read_excel(plan_name)

Selected_data = combined_df[["Date", "CASE_ID","Repair Call Status","DEPENDENCY","CALL_AGEING","STATUS_AGEING","PLAN_NAME","Vas PolicyId"]].copy()
Selected_data.fillna(0,inplace=True)
merged_df = pd.merge(Selected_data, df_plan, on="PLAN_NAME", how ="left" )
Aging_merged_df = pd.merge(merged_df,df_aging , on ="Repair Call Status" , how ="left")
Aging_merged_df["Breach_status"] = Aging_merged_df.apply(lambda row: 'Breached' if row["Design SLA"] < row["STATUS_AGEING"] else 'Not_Breached', axis=1)
Aging_merged_df["Deep_Pain"] = Aging_merged_df.apply(lambda row: "Deep_Pain" if row["Deep pain SLA"] < row["STATUS_AGEING"] else ("Pain" if row["Design SLA"] < row["STATUS_AGEING"] else "Not_Breached"), axis=1 )



summary_grouped = Aging_merged_df.groupby(["Breach_status","Date","Process"], as_index=False)["CASE_ID"].count()
summary_grouped.rename(columns={"CASE_ID": "Total_breach"}, inplace= True)

processes = ['PL','VAS']
pivot_data_dict = {}
pivot_percentrage_dict = {}

for Process in processes:
    filtered_data = summary_grouped[summary_grouped["Process"]== Process]
    ##filtered_data = summary_grouped[summary_grouped["Process"].isin(["PL"])]##

pivot_data = filtered_data[filtered_data["Index"].isin(["PL","VAS"])].pivot_table(
index= ["Breach_status", "Process"],
columns="Date",
values = "Total_breach",
fill_value = 0    
).reset_index()

pivot_data.iloc[:, 2:] = pivot_data.iloc[:, 2:].apply(pd.to_numeric)


pivot_numeric = pivot_data.iloc[:, 2:]  # Select numeric columns
pivot_percentage = pivot_numeric.div(pivot_numeric.sum(axis=0), axis=1) * 100

# Add back non-numeric columns
pivot_data_percentage = pivot_data.copy()
pivot_data_percentage.iloc[:, 2:] = pivot_percentage

# Add Grand Total row (100% for each column)
grand_total = pd.DataFrame([["Grand_total", "", *[100] * (pivot_data.shape[1] - 2)]], columns=pivot_data_percentage.columns)
pivot_data_percentage = pd.concat([pivot_data_percentage, grand_total], ignore_index=True)

print(pivot_data)
print(pivot_data_percentage)
    
output_path = r"C:\Users\rajeshkumar.t\Desktop\BBD\Breach_data.xlsx"
with pd.ExcelWriter(output_path, engine ="xlsxwriter") as writer:
    pivot_data.to_excel(writer, sheet_name = "pivot data" , index= False)
    pivot_data_percentage.to_excel(writer,sheet_name = "Percentage Data" , index= False)
    print(f"Excel save file at :{output_path}")


    Breach_status        Date Process  Total_breach
0        Breached  2025-02-01  Others          1463
1        Breached  2025-02-01      PL           823
2        Breached  2025-02-01     VAS           157
3        Breached  2025-02-02  Others          1522
4        Breached  2025-02-02      PL           908
..            ...         ...     ...           ...
511  Not_Breached  2025-04-29      PL          5472
512  Not_Breached  2025-04-29     VAS          1038
513  Not_Breached  2025-04-30  Others          1001
514  Not_Breached  2025-04-30      PL          5144
515  Not_Breached  2025-04-30     VAS          1071

[516 rows x 4 columns]
Date Breach_status Process  2025-02-01  2025-02-02  2025-02-03  2025-02-04  \
0         Breached      PL       823.0       908.0      1060.0       856.0   
1     Not_Breached      PL      2484.0      2429.0      2348.0      2720.0   

Date  2025-02-05  2025-02-06  2025-02-07  2025-02-08  ...  2025-04-21  \
0          528.0       902.0       701.0    